In [1]:
%pip install -qU google-generativeai
%pip install -qU google-ai-generativelanguage==0.6.15
%pip install -qU langchain-google-genai
%pip install -qU langchain-community
%pip install -qU langgraph
%pip install -qU langgraph langchain-community
%pip install -qU python-dotenv

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
#SE IMPORTAN MODULOS 
import os #Variables de entorno
import re 
import google.genai as genai #llm
from langgraph.graph import StateGraph, END #modulo para hacer graphos (arbol de decision con nodos)
from typing import TypedDict #para crear una clase y darle formato a los diccionarios

In [5]:
#Se cargan las API mediante las variables de entrono .env
#nos conectamos a las llm y declaramos el modelo
from dotenv import load_dotenv
import google.generativeai as genai 

load_dotenv()

GOOGLE_API_KEY = os.getenv('GEMINI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')
response = model.generate_content("Hola mundo")

print(response.text)

¡Hola! 😊


In [6]:
#Creacion del agente 
class Agent: #se inicializa la clase agente 
    def __init__(self, system=""): #funcion que inicia, recibe el promt de sistema en ""
        self.system = system #aqui se almacena el promt de sistema
        self.messages = [] #lista vacia con los mensajes 
        if self.system:
            self.messages.append({"role": "system", "content": system}) #se hace un append en la lista con todos los promts de sistema

    def __call__(self, message): #funcion del llamado, recibe mensaje
        self.messages.append({"role": "user", "content": message}) 
        result = self.execute(message) #ejecuta la funcion de abajo
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self,message): #esta funcion activa el llm
        client = genai.GenerativeModel("gemini-2.5-flash")
        completion = client.generate_content(message) #genera contenido a partir del mensaje de entrada
        return completion.text

In [7]:
#probamos al agente 
agente = Agent(system="Eres un asistente util y objetivo.") #llamamos a la clase y le damos un promt de sistema
print(agente("Cual es la capital de Mexico")) #ahora se invocar a la variable creada en la linea anterior en donde se pone el request 

La capital de México es la **Ciudad de México**.


In [8]:
#se le da el promt que recomienda la pagina para correcto funcionamiento del agente 
PROMPT_REACT = """
Funcionas en un ciclo de Pensamiento, Acción, Pausa y Observación.
Al final del ciclo, proporcionas una Respuesta.
Usa "Pensamiento" para describir tu razonamiento.
Usa "Acción" para ejecutar herramientas - y luego retorna "PAUSA".
La "Observación" será el resultado de la acción ejecutada.
Acciones disponibles:

consultar_stock: devuelve la cantidad disponible de un artículo en el inventario (ej: "consultar_stock: teclado")

consultar_precio_producto: devuelve el precio unitario de un producto (ej: "consultar_precio_producto: mouse gamer")

Ejemplo:
Pregunta: ¿Cuántos monitores tenemos en el inventario?
Pensamiento: Debo consultar la acción consultar_stock para saber la cantidad de monitores.
Acción: consultar_stock: monitor
PAUSA

Observación: Tenemos 75 monitores en el inventario.
Respuesta: Hay 75 monitores en el inventario.
""".strip()

In [9]:
#creacion del estado del agente 
#se debe instanciar el estado en el cual el agente trabaja
#basicamente es una clase que define el formato de las input y outputs
#Typedict es la herencia que define inputs y/o respuesta en forma de diccionario
class EstadoAgente(TypedDict):
    pregunta: str
    historial: list[str] #aqui sera un historial de preguntas y respuesta en forma de lista/str/dict
    accion_pendiente: str
    respuesta_final: str

In [10]:
#Se definen los agentes
def consultar_stock(item: str) -> str: #recibe str y devuelve str 
    #se le da promt al agente ya que lo leera
    """
    Simula la consulta de stock de item en el inventario.
    """
    item = item.lower()
    #se le da ejemplo de stock
    stock = {
        "monitor": 75,
        "teclado": 120,
        "mouse de gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impresora": 15
    }

    #si el item dado como entrada a la funcion esta en el diccionario stock entonces
    if item in stock:
        return f"Tenemos {stock[item]} {item}s en stock." #se retorna el mensaje
    else:
        return f"Item '{item}' no encontrado en el inventario."

#funcion muy parecida a la anterior, solo que ahora precio 
def consultar_precio_producto(producto: str) -> str:
    """
    Simula la consulta del precio unitario de un producto.
    """
    producto = producto.lower()
    precios = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse de gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impresora": 750.00
    }

    if producto in precios:
        return f"El precio de un(a) {producto} es USD {precios[producto]:.2f}."
    else:
        return f"producto '{producto}' no hallado en la lista de precios."

In [11]:
print(consultar_stock("teclado"))
print(consultar_precio_producto("impresora"))
print(consultar_stock("monitor"))
print(consultar_stock("sillas"))

Tenemos 120 teclados en stock.
El precio de un(a) impresora es USD 750.00.
Tenemos 75 monitors en stock.
Item 'sillas' no encontrado en el inventario.
